## Build the Neural Network

* Neural networks comprise of layers/modules that perform operations on data
* torch.nn namespace provides all the building blocks you need to build your own neural network
    * Every module in PyTorch subclasses the nn.Module
* A neural network is a module itself that consists of other modules (layers)  
=> This nested structure allows for building and managing complex architectures easily

In [1]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

### Get Device for Training
* If the current accelerator(such as CUDA, MPS, MTIA, or XPU) is available, we will use it
* Otherwise, we use the CPU

In [5]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


### Define the Class
* We define our neural network by subclassing `nn.Module`
    * Every `nn.Module` subclass implements the operations on input data in the `forward` method
* Initialize the neural network layers in `__init__`

In [3]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )
    
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [14]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


* To use model, we pass it the input data
* This executes the model's `forward`, along with some background operations
    * Don't call `model.forward` directly

* Calling the model on the input returns a 2-dimensional tensor with dim=0 corresponding to each output of 10 raw predicted values for each class
* dim=1 corresponding to the individual values of each output
* We get the prediction probabilities by passing it through an instance of the `nn.Softmax` module

In [22]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

Predicted class: tensor([3])


### Model Layers

In [23]:
input_image = torch.rand(3, 28, 28)
print(input_image.size())

torch.Size([3, 28, 28])


#### nn.Flatten
* Initialize the nn.Flatten layer to convert each 2D 28x28 image into a contiguous array of 784 pixel values
    * The minibatch dimension (at dim=0) is maintained

In [26]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


#### nn.Linear
* The linear layer is a module that applies a linear transformation on the input using its stored weights and biases

In [27]:
layer1 = nn.Linear(in_features=28 * 28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


#### nn.ReLU
* Non-linear activations are what create the complex mappings between the model’s inputs and outputs
* They are applied after linear transformations to introduce nonlinearity, helping neural networks learn a wide variety of phenomena

In [28]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[ 4.1480e-01, -6.3898e-02, -2.8972e-01,  1.3296e-01,  2.9940e-01,
          1.5387e-01,  1.3558e-02,  3.7960e-01,  6.6409e-01,  1.4155e-01,
         -1.7567e-01,  1.3837e-01,  6.0102e-01, -9.9388e-01, -6.6729e-02,
          2.6379e-01,  5.8537e-01,  1.1725e-01, -1.4402e-01,  3.1601e-01],
        [ 2.8440e-01, -9.9463e-02, -7.7778e-01,  2.5003e-01, -1.1596e-01,
          2.0918e-04, -2.8790e-01,  9.9204e-02,  4.4187e-01, -1.4127e-01,
         -6.4055e-02,  1.8583e-02,  2.3174e-01, -1.0351e+00,  5.2218e-02,
         -1.1816e-01,  7.6838e-01,  1.7814e-01, -5.3912e-01,  1.3398e-01],
        [ 5.5499e-01, -1.8994e-01, -4.3689e-01,  7.2752e-01,  3.2311e-01,
         -1.3055e-01, -2.2253e-01,  2.6499e-01,  5.2169e-01, -1.7339e-01,
         -1.8043e-01,  2.8864e-01,  5.2028e-01, -9.2968e-01, -2.3152e-02,
          1.1310e-01,  4.8065e-01,  2.9269e-01, -2.5186e-01,  6.1218e-01]],
       grad_fn=<AddmmBackward0>)


After ReLU: tensor([[4.1480e-01, 0.0000e+00, 0.0000e+00, 1.3

#### nn.Sequential
* nn.Sequential is an ordered container of modules
* The data is passed through all the modules in the same order as defined
* You can use sequential containers to put together a quick network like `seq_modules`

In [29]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)

#### nn.Softmax
* The logits are scaled to values [0, 1] representing the model’s predicted probabilities for each class
* `dim` parameter indicates the imension along which the values must sum to 1

In [30]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)

### Model Parameters
* Many layers inside a neural network are parameterized
    * weights and biases that are optimized during training
* Subclassing `nn.Module` automatically tracks all fields defined inside your model object, and makes all parameters accessible using your model’s `parameters()` or `named_parameters()` methods

In [31]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[ 0.0317,  0.0060,  0.0152,  ..., -0.0027, -0.0257,  0.0035],
        [-0.0343,  0.0044,  0.0244,  ...,  0.0012,  0.0092, -0.0195]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([-0.0024,  0.0245], grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[ 0.0327,  0.0341,  0.0152,  ...,  0.0160, -0.0050,  0.0320],
        [-0.0438,  0.0415,  0.0078,  ..., -0.0411, -0.0204, -0.0301]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.bias | 